# 05 BERT Text Model — Colab Clean

This notebook trains the DistilBERT embedding + Logistic Regression model using the shared train/val/test splits and saves Week 3 fusion-ready probabilities and embeddings.

In [1]:
# Fresh clone. Skip this cell if the repo is already present and up to date.
!rm -rf /content/movie_genre_classification
!git clone https://github.com/oluwoleadetifa/movie_genre_classification.git
%cd /content/movie_genre_classification

import sys
sys.path.append("/content/movie_genre_classification")

Cloning into 'movie_genre_classification'...
remote: Enumerating objects: 324, done.
remote: Counting objects: 100% (139/139), done.
remote: Compressing objects: 100% (121/121), done.
remote: Total 324 (delta 72), reused 44 (delta 18), pack-reused 185 (from 1)
Receiving objects: 100% (324/324), 1.46 MiB | 12.47 MiB/s, done.
Resolving deltas: 100% (162/162), done.
/content/movie_genre_classification


In [2]:
!pip install -q transformers torch scikit-learn pandas numpy tqdm joblib

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from pathlib import Path
import shutil
import os

PROJECT_DIR = Path("/content/movie_genre_classification")
DRIVE_BASE = Path("/content/drive/MyDrive/Movie_Genre_Project")

# Expected Drive locations.
SPLITS_SRC = DRIVE_BASE / "splits"
PROCESSED_SRC = DRIVE_BASE / "dataset_processed" / "movies_with_posters.csv"

SPLITS_DST = PROJECT_DIR / "data" / "splits"
PROCESSED_DST = PROJECT_DIR / "data" / "processed" / "movies_with_posters.csv"

SPLITS_DST.mkdir(parents=True, exist_ok=True)
PROCESSED_DST.parent.mkdir(parents=True, exist_ok=True)

# Copy shared splits only. Do not regenerate random splits.
if SPLITS_SRC.exists():
    for name in ["train.csv", "val.csv", "test.csv"]:
        shutil.copy2(SPLITS_SRC / name, SPLITS_DST / name)
else:
    print("Drive splits folder not found. Using existing repo splits if present:", SPLITS_DST)

if PROCESSED_SRC.exists():
    shutil.copy2(PROCESSED_SRC, PROCESSED_DST)
else:
    print("Processed dataset not found in Drive. Continuing if splits already contain required columns.")

print("Splits:")
!ls -lh /content/movie_genre_classification/data/splits

Splits:
total 1.6M
-rw------- 1 root root 269K Apr 22 02:30 test.csv
-rw------- 1 root root 1.1M Apr 22 02:31 train.csv
-rw------- 1 root root 196K Apr 22 02:31 val.csv


In [5]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import joblib

PROJECT_DIR = Path("/content/movie_genre_classification")
SPLITS_DIR = PROJECT_DIR / "data" / "splits"
OUTPUT_DIR = PROJECT_DIR / "outputs"
METRICS_DIR = OUTPUT_DIR / "metrics"
MODELS_DIR = OUTPUT_DIR / "models"
PROBS_DIR = OUTPUT_DIR / "probs"
FEATURES_DIR = OUTPUT_DIR / "features"

for d in [METRICS_DIR, MODELS_DIR, PROBS_DIR, FEATURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

TEXT_COLUMN = "description"
LABEL_COLUMN = "label"
CLASS_NAMES = ["action", "comedy", "horror", "romance"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [6]:
train_df = pd.read_csv(SPLITS_DIR / "train.csv")
val_df = pd.read_csv(SPLITS_DIR / "val.csv")
test_df = pd.read_csv(SPLITS_DIR / "test.csv")

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)
print("Columns:", train_df.columns.tolist())

assert TEXT_COLUMN in train_df.columns, f"Missing text column: {TEXT_COLUMN}"
assert LABEL_COLUMN in train_df.columns, f"Missing label column: {LABEL_COLUMN}"

# Keep the shared split row order unchanged.
for df in [train_df, val_df, test_df]:
    df[TEXT_COLUMN] = df[TEXT_COLUMN].fillna("").astype(str)

Train: (480, 6)
Val: (103, 6)
Test: (104, 6)
Columns: ['movie_id', 'description', 'genre', 'image_path', 'image_exists', 'label']


In [7]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
bert_model = AutoModel.from_pretrained("distilbert-base-uncased").to(device)
bert_model.eval()

@torch.no_grad()
def encode_texts(texts, batch_size=16, max_length=256):
    embeddings = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = list(texts[i:i + batch_size])

        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        encoded = {k: v.to(device) for k, v in encoded.items()}
        outputs = bert_model(**encoded)

        # DistilBERT [CLS] token embedding.
        cls_embeddings = outputs.last_hidden_state[:, 0, :]
        embeddings.append(cls_embeddings.cpu().numpy())

    return np.vstack(embeddings)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
X_train_bert = encode_texts(train_df[TEXT_COLUMN].tolist(), batch_size=16)
X_val_bert = encode_texts(val_df[TEXT_COLUMN].tolist(), batch_size=16)
X_test_bert = encode_texts(test_df[TEXT_COLUMN].tolist(), batch_size=16)

y_train = train_df[LABEL_COLUMN].values
y_val = val_df[LABEL_COLUMN].values
y_test = test_df[LABEL_COLUMN].values

print("BERT train embeddings:", X_train_bert.shape)
print("BERT val embeddings:", X_val_bert.shape)
print("BERT test embeddings:", X_test_bert.shape)

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/7 [00:00<?, ?it/s]

  0%|          | 0/7 [00:00<?, ?it/s]

BERT train embeddings: (480, 768)
BERT val embeddings: (103, 768)
BERT test embeddings: (104, 768)


In [9]:
# Save embeddings for Week 3 intermediate fusion.
np.save(FEATURES_DIR / "bert_train_embeddings.npy", X_train_bert)
np.save(FEATURES_DIR / "bert_val_embeddings.npy", X_val_bert)
np.save(FEATURES_DIR / "bert_test_embeddings.npy", X_test_bert)

np.save(PROBS_DIR / "y_train.npy", y_train)
np.save(PROBS_DIR / "y_val.npy", y_val)
np.save(PROBS_DIR / "y_test.npy", y_test)

print("Saved BERT embeddings and labels.")

Saved BERT embeddings and labels.


In [10]:
clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(X_train_bert, y_train)

joblib.dump(clf, MODELS_DIR / "bert_logreg.joblib")

print("Saved Logistic Regression model.")
print("Classifier class order:", clf.classes_)

Saved Logistic Regression model.
Classifier class order: [0 1 2 3]


In [11]:
y_pred = clf.predict(X_test_bert)
test_probs = clf.predict_proba(X_test_bert)
val_probs = clf.predict_proba(X_val_bert)
train_probs = clf.predict_proba(X_train_bert)

test_acc = accuracy_score(y_test, y_pred)
test_macro_f1 = f1_score(y_test, y_pred, average="macro")

print("BERT Test Accuracy:", round(test_acc, 4))
print("BERT Test Macro F1:", round(test_macro_f1, 4))
print(classification_report(y_test, y_pred, zero_division=0))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

BERT Test Accuracy: 0.7692
BERT Test Macro F1: 0.7369
              precision    recall  f1-score   support

           0       0.79      0.83      0.81        36
           1       0.55      0.69      0.61        16
           2       0.88      0.79      0.83        38
           3       0.75      0.64      0.69        14

    accuracy                           0.77       104
   macro avg       0.74      0.74      0.74       104
weighted avg       0.78      0.77      0.77       104

Confusion Matrix:
[[30  3  3  0]
 [ 2 11  1  2]
 [ 4  3 30  1]
 [ 2  3  0  9]]


In [12]:
# Save probabilities for Week 3 late fusion.
np.save(PROBS_DIR / "bert_train_probs.npy", train_probs)
np.save(PROBS_DIR / "bert_val_probs.npy", val_probs)
np.save(PROBS_DIR / "bert_test_probs.npy", test_probs)
np.save(PROBS_DIR / "bert_test_preds.npy", y_pred)

results_bert = {
    "test_accuracy": float(test_acc),
    "test_macro_f1": float(test_macro_f1),
    "class_order": clf.classes_.tolist(),
    "classification_report": classification_report(y_test, y_pred, output_dict=True, zero_division=0),
    "confusion_matrix": confusion_matrix(y_test, y_pred).tolist()
}

with open(METRICS_DIR / "bert_results.json", "w") as f:
    json.dump(results_bert, f, indent=2)

with open(OUTPUT_DIR / "class_order_bert.json", "w") as f:
    json.dump({"class_names": clf.classes_.tolist()}, f, indent=2)

print("Saved BERT results, probabilities, embeddings, and model.")
print("Probability shape:", test_probs.shape)

Saved BERT results, probabilities, embeddings, and model.
Probability shape: (104, 4)


In [13]:
# Optional: copy outputs back to Drive so they survive Colab runtime reset.
DRIVE_OUTPUT_DIR = DRIVE_BASE / "outputs"
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

shutil.copytree(OUTPUT_DIR, DRIVE_OUTPUT_DIR, dirs_exist_ok=True)
print("Copied outputs to:", DRIVE_OUTPUT_DIR)

Copied outputs to: /content/drive/MyDrive/Movie_Genre_Project/outputs
